## Example 1: Basic ptyprocess functionality and PTY creation

In [ ]:
import ptyprocess
import os
import select

# Example 1: Basic ptyprocess functionality and PTY creation
print("ptyprocess module imported successfully")

try:
    print("Example 1 - ptyprocess PTY creation and basic operations")
    
    # Show ptyprocess version and main classes
    print(f"ptyprocess version: {ptyprocess.__version__}")
    print(f"PtyProcess class: {ptyprocess.PtyProcess}")
    
    # PTY process attributes and methods
    pty_methods = [
        "spawn", "kill", "terminate", "wait", "read", "write",
        "readline", "isalive", "exitstatus", "pid"
    ]
    
    print("PtyProcess methods:")
    for method in pty_methods:
        if hasattr(ptyprocess.PtyProcess, method):
            print(f"  ✓ {method}")
        else:
            print(f"  - {method} (not found)")
    
    # Simple command spawning (echo command)
    print("Testing PTY process creation:")
    # Use a simple command that exists on most systems
    child = ptyprocess.PtyProcess.spawn(['echo', 'Hello PTY'])
    
    print(f"Process PID: {child.pid}")
    print(f"Process alive: {child.isalive()}")
    
    # Read output. PtyProcess.read() takes no timeout (that is pexpect's layer), so poll the
    # PTY fd with select() and read until the child closes it (EOFError).
    output = b""
    while True:
        ready, _, _ = select.select([child.fd], [], [], 5)
        assert ready, "no output from the PTY within 5 s"
        try:
            output += child.read(1000)
        except EOFError:
            break
    print(f"Process output: {output.decode().strip()}")
    assert output.decode().strip() == "Hello PTY", f"unexpected PTY output {output!r}"
    
    # Wait for completion
    child.wait()
    print(f"Exit status: {child.exitstatus}")
    assert child.exitstatus == 0, f"echo exited {child.exitstatus}"
    
    print("ptyprocess example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"ptyprocess example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Interactive shell and command execution

In [ ]:
import ptyprocess
import re
import select
import time

# Example 2: Interactive shell and command execution
print("ptyprocess module imported successfully")

try:
    print("Example 2 - ptyprocess interactive shell operations")
    
    # Spawn a shell process
    # Use sh which is available on Unix-like systems
    shell = ptyprocess.PtyProcess.spawn(['sh'])
    print(f"Shell process spawned with PID: {shell.pid}")
    print(f"Shell alive: {shell.isalive()}")
    
    # Send commands to shell
    commands = [
        b'echo "Hello from shell"\n',
        b'pwd\n', 
        b'date\n',
        b'exit\n'
    ]
    
    transcript = b""
    for cmd in commands:
        print(f"Sending command: {cmd.decode().strip()}")
        shell.write(cmd)
        time.sleep(0.1)  # Give command time to execute
        
        # Read whatever the shell answered. ptyprocess has no TIMEOUT and read() no timeout=
        # (both are pexpect's): poll the PTY fd, and stop at EOF once `exit` closes it.
        while select.select([shell.fd], [], [], 0.5)[0]:
            try:
                response = shell.read(1000)
            except EOFError:
                break
            transcript += response
            print(f"Response: {response.decode(errors='replace').strip()}")
    
    # Wait for shell to exit
    shell.wait()
    print(f"Shell exit status: {shell.exitstatus}")
    assert shell.exitstatus == 0, f"sh exited {shell.exitstatus}"
    # The PTY echoes each command back, so match the OUTPUT line (starts a line), not the echoed `echo "..."`.
    assert re.search(rb"[\r\n]Hello from shell\r\n", transcript), f"echo output missing from {transcript!r}"
    
    # Show PTY-specific features
    print("PTY features:")
    print("- Full terminal emulation")
    print("- Signal handling (Ctrl+C, etc.)")
    print("- Terminal size control")
    print("- Raw/cooked mode support")
    print("- Interactive program control")
    
    # PTY vs subprocess differences
    print("PTY advantages over subprocess:")
    print("- Programs behave as if running in terminal")
    print("- Interactive applications work correctly")
    print("- Color output and terminal features preserved")
    print("- Better handling of password prompts")
    
    print("ptyprocess example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"ptyprocess example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Process control and signal handling

In [ ]:
import ptyprocess
import signal
import os

# Example 3: Process control and signal handling
print("ptyprocess module imported successfully")

try:
    print("Example 3 - ptyprocess process control and signals")
    
    # Spawn a long-running process for control demonstration
    try:
        # Use a simple sleep command
        sleeper = ptyprocess.PtyProcess.spawn(['sleep', '10'])
        print(f"Sleep process PID: {sleeper.pid}")
        print(f"Process alive: {sleeper.isalive()}")
        
        # Process state information
        print(f"File descriptor: {sleeper.fd}")
        print(f"Child fd: {sleeper.child_fd if hasattr(sleeper, 'child_fd') else 'N/A'}")
        
        # Demonstrate termination
        print("Terminating process...")
        sleeper.terminate()
        
        # Check if terminated
        print(f"Process alive after terminate: {sleeper.isalive()}")
        
        # Wait for process to finish
        sleeper.wait()
        print(f"Final exit status: {sleeper.exitstatus}")
        
    except Exception as e:
        print(f"Process control error: {e}")
        raise
    
    # Signal handling demonstration
    signals_demo = {
        "SIGTERM": "Graceful termination",
        "SIGKILL": "Forced termination", 
        "SIGINT": "Interrupt (Ctrl+C)",
        "SIGSTOP": "Suspend process",
        "SIGCONT": "Resume process"
    }
    
    print("Signal handling capabilities:")
    for sig, desc in signals_demo.items():
        if hasattr(signal, sig):
            sig_num = getattr(signal, sig)
            print(f"  {sig} ({sig_num}): {desc}")
    
    # Process lifecycle methods
    lifecycle_methods = {
        "spawn": "Create new PTY process",
        "kill": "Send signal to process",
        "terminate": "Gracefully terminate",
        "wait": "Wait for process completion",
        "isalive": "Check if process is running"
    }
    
    print("Process lifecycle methods:")
    for method, desc in lifecycle_methods.items():
        print(f"  {method}(): {desc}")
    
    # Error handling patterns
    print("Error handling:")
    print("- EOFError from read() when the child closes the PTY")
    print("- select() on proc.fd for reads with a timeout")
    print("- Check exitstatus for process result")
    print("- Handle broken pipe on write operations")
    
    print("ptyprocess example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"ptyprocess example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Terminal size control and environment handling

In [ ]:
import ptyprocess
import select

# Example 4: Terminal size control and environment handling
print("ptyprocess module imported successfully")

try:
    print("Example 4 - ptyprocess terminal size and environment")
    
    # Terminal size control
    try:
        # Spawn process with specific terminal dimensions
        print("Testing terminal size control:")
        
        # Default spawn
        proc = ptyprocess.PtyProcess.spawn(['echo', 'Terminal test'])
        print(f"Process created with default terminal size")
        
        # Check terminal size methods
        if hasattr(proc, 'getwinsize'):
            try:
                rows, cols = proc.getwinsize()
                print(f"Terminal size: {rows} rows x {cols} columns")
            except:
                print("Could not get terminal size")
                raise
        
        if hasattr(proc, 'setwinsize'):
            try:
                proc.setwinsize(50, 120)  # 50 rows, 120 columns
                print("Terminal size set to 50x120")
            except:
                print("Could not set terminal size")
                raise
        
        # Read output and clean up. PtyProcess.read() takes no timeout (ptyprocess has no TIMEOUT;
        # both are pexpect's), so poll the PTY fd and read until the child closes it.
        output = b""
        while True:
            ready, _, _ = select.select([proc.fd], [], [], 5)
            assert ready, "no output from the PTY within 5 s"
            try:
                output += proc.read(100)
            except EOFError:
                break
        print(f"Output: {output.decode().strip()}")
        assert output.decode().strip() == "Terminal test", f"unexpected PTY output {output!r}"
        
        proc.wait()
        assert proc.exitstatus == 0, f"echo exited {proc.exitstatus}"
        
    except Exception as e:
        print(f"Terminal size error: {e}")
        raise
    
    # Environment variable handling
    print("Environment variable handling:")
    
    # Custom environment
    custom_env = {
        'PATH': '/usr/bin:/bin',
        'HOME': '/tmp',
        'TERM': 'xterm-256color',
        'LANG': 'en_US.UTF-8'
    }
    
    print("Custom environment variables:")
    for key, value in custom_env.items():
        print(f"  {key}={value}")
    
    # Working directory control
    print("Working directory control:")
    print("- Can specify cwd parameter in spawn()")
    print("- Process inherits environment by default")
    print("- Custom environment can be provided")
    
    # Spawn parameters demonstration
    spawn_params = {
        "argv": "Command and arguments list",
        "cwd": "Working directory path",
        "env": "Environment variables dict", 
        "echo": "Echo input characters",
        "preexec_fn": "Function to run before exec",
        "dimensions": "Terminal size (rows, cols)"
    }
    
    print("PtyProcess.spawn() parameters:")
    for param, desc in spawn_params.items():
        print(f"  {param}: {desc}")
    
    # Example spawn with parameters
    spawn_example = '''
    proc = PtyProcess.spawn(
        argv=['bash'],
        cwd='/tmp',
        env={'TERM': 'xterm', 'PATH': '/usr/bin'},
        dimensions=(24, 80)
    )
    '''
    print("Spawn with parameters example:")
    print(spawn_example)
    
    print("ptyprocess example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"ptyprocess example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Real-world applications and best practices

In [ ]:
import ptyprocess

# Example 5: Real-world applications and best practices
print("ptyprocess module imported successfully")

try:
    print("Example 5 - ptyprocess real-world applications and best practices")
    
    # Common use cases
    use_cases = {
        "Interactive shells": "SSH clients, terminal emulators",
        "Password prompts": "sudo, ssh-keygen, database connections",
        "Progress indicators": "rsync, wget, package installers", 
        "Terminal applications": "vi, emacs, htop, interactive CLIs",
        "Automated testing": "Terminal-based application testing",
        "Process automation": "Scripting interactive command-line tools"
    }
    
    print("Common ptyprocess use cases:")
    for use_case, examples in use_cases.items():
        print(f"  {use_case}: {examples}")
    
    # Best practices
    best_practices = [
        "Handle EOFError when the child closes the PTY",
        "Use select() on proc.fd for timed reads",
        "Close processes explicitly with close() or terminate()",
        "Handle process death gracefully",
        "Use non-blocking reads when possible",
        "Set terminal size for applications that need it",
        "Properly handle encoding/decoding of text"
    ]
    
    print("Best practices:")
    for practice in best_practices:
        print(f"  ✓ {practice}")
    
    # Error handling pattern
    error_handling_pattern = '''
    import select
    from ptyprocess import PtyProcess

    proc = PtyProcess.spawn(['interactive_app'])
    try:
        while True:
            # read() takes no timeout: wait on the PTY file descriptor with select().
            ready, _, _ = select.select([proc.fd], [], [], 1.0)
            if not ready:
                continue  # nothing to read yet
            try:
                data = proc.read(1024)
            except EOFError:
                break  # the child closed the PTY
            process_output(data)
    finally:
        proc.close(force=True)  # close the PTY and terminate the child if it is still running
        proc.wait()  # reap the child
    '''
    print("Robust error handling pattern:")
    print(error_handling_pattern)
    
    # Performance considerations
    print("Performance considerations:")
    print("- PTY overhead is higher than regular subprocess")
    print("- Use appropriate buffer sizes for reading")
    print("- Consider process pooling for high-frequency operations")
    print("- Monitor file descriptor limits in long-running applications")
    print("- Clean up processes to avoid resource leaks")
    
    # Integration with other libraries
    integrations = {
        "pexpect": "Higher-level automation built on ptyprocess",
        "paramiko": "SSH automation with PTY support",
        "fabric": "Remote command execution",
        "asyncio": "Asynchronous process management"
    }
    
    print("Integration with other libraries:")
    for lib, desc in integrations.items():
        print(f"  {lib}: {desc}")
    
    print("ptyprocess example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"ptyprocess example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")